<h3>Batch processing notebook based on 3D Cellpose dev logic (no napari visualization)</h3>

In [ ]:
from pathlib import Path
from tqdm import tqdm
import czifile
import numpy as np
import tifffile
import pandas as pd
from skimage.measure import regionprops_table
import plotly.express as px
import pyclesperanto_prototype as cle

from utils import (
    extract_scaling_metadata,
    predict_nuclei_labels,
    simulate_cytoplasm,
    _keep_objects_in_size_range,
)
from io_utils import (
    list_images,
    ensure_output_dir,
    load_precomputed_results_if_available,
)

cle.select_device("RTX")

### Configure experiment inputs
Set your input directory and analysis parameters.

In [ ]:
# Path where .czi images are stored
RAW_DATA_DIRECTORY = "./raw_data/WT_organoids/"
#RAW_DATA_DIRECTORY = "./raw_data/20240703 PGE2MSN Tum_organoids"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# YAP intensity channel used for nuclei/cytoplasm measurements
YAP_CHANNEL = 1

# Inclusive min/max nuclei volume used to filter predicted labels
# Set to None to skip label-size filtering
MIN_MAX_NUCLEI_VOLUME = (150,2000)

# Cytoplasm thickness in voxels (starting from the nucleus)
# If padding is applied, the thickness will be applied to the padded nucleus
CYTOPLASM_THICKNESS = 1

# Padding to keep a safe distance between nuclei and cytoplasm borders
# It expands the nucleus by the padding value in all directions
NUCLEI_PADDING = 1

directory_path = Path(RAW_DATA_DIRECTORY)
input_folder_id = directory_path.stem
images = list_images(directory_path, "czi")
len(images)

In [ ]:
# Prepare output directories
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, input_folder_id, results_type="nuclei_labels")
bp_results_dir = ensure_output_dir("./results", input_folder_id, results_type="bp_results")

dataframes = []

for image in tqdm(images):
    # Read path storing raw image and extract filename
    file_path = Path(image)
    filename = file_path.stem

    # Read the image file and remove singleton dimensions
    img = czifile.imread(image)
    img = img.squeeze()

    # Extract experiment, mouse, treatment and replica ids
    experiment_id = filename.split(" ")[0]
    mouse_id = filename.split(" ")[1]
    treatment_id = filename.split(" ")[2]
    replica_id = filename.split(" ")[-1]

    # Use YAP channel for intensity-based measurements
    yap_stack = img[YAP_CHANNEL, :, :, :]

    # Calculate anisotropy correction factor for Cellpose 3D inference
    scaling_x_um, scaling_y_um, scaling_z_um = extract_scaling_metadata(file_path)
    rescale_factor = scaling_z_um / np.mean([scaling_x_um, scaling_y_um])

    # Load existing labels, otherwise predict and save them
    nuclei_labels = load_precomputed_results_if_available(
        nuclei_labels_dir, filename, results_type="nuclei_labels"
    )

    if nuclei_labels is not None:
        if MIN_MAX_NUCLEI_VOLUME is not None:
            nuclei_labels = _keep_objects_in_size_range(nuclei_labels, MIN_MAX_NUCLEI_VOLUME)
    else:
        nuclei_labels = predict_nuclei_labels(
            img,
            rescale_factor,
            NUCLEI_CHANNEL,
            MIN_MAX_NUCLEI_VOLUME,
            visualize=False,
        )
        nuclei_labels_path = nuclei_labels_dir / f"{filename}_nuclei_labels.tif"
        tifffile.imwrite(nuclei_labels_path, nuclei_labels)

    # Simulate cytoplasm labels around each nucleus
    cytoplasm = simulate_cytoplasm(nuclei_labels, cytoplasm_thickness=CYTOPLASM_THICKNESS, nuclei_padding=NUCLEI_PADDING)

    # Extract regionprops from nuclei and simulated-cytoplasm labels
    props_nuclei = regionprops_table(
        label_image=nuclei_labels,
        intensity_image=yap_stack,
        properties=["label", "intensity_mean", "area"],
    )
    props_cytoplasm = regionprops_table(
        label_image=cytoplasm,
        intensity_image=yap_stack,
        properties=["label", "intensity_mean", "area"],
    )

    # Build and merge per-label dataframes
    df_nuclei = pd.DataFrame(props_nuclei)
    df_cytoplasm = pd.DataFrame(props_cytoplasm)

    df_nuclei.rename(columns={"intensity_mean": "intensity_mean_nuclei", "area": "area_nuclei"}, inplace=True)
    df_cytoplasm.rename(columns={"intensity_mean": "intensity_mean_cyto", "area": "area_cyto"}, inplace=True)

    merged_df = pd.merge(df_nuclei, df_cytoplasm, on="label")

    # Calculate nuclei/cytoplasm ratio of mean YAP intensity
    merged_df["nuclei_cyto_ratio"] = merged_df["intensity_mean_nuclei"] / merged_df["intensity_mean_cyto"]

    # Add image metadata ids
    merged_df.insert(0, "experiment_id", experiment_id)
    merged_df.insert(1, "mouse_id", mouse_id)
    merged_df.insert(2, "treatment_id", treatment_id)
    merged_df.insert(3, "replica_id", replica_id)
    if MIN_MAX_NUCLEI_VOLUME is not None:
        merged_df.insert(4, "min_nuc_vol", MIN_MAX_NUCLEI_VOLUME[0])
        merged_df.insert(5, "max_nuc_vol", MIN_MAX_NUCLEI_VOLUME[1])
    else:
        merged_df.insert(4, "min_nuc_vol", None)
        merged_df.insert(5, "max_nuc_vol", None)
    merged_df.insert(6, "cyto_thickness", CYTOPLASM_THICKNESS)
    merged_df.insert(7, "nuclei_padding", NUCLEI_PADDING)

    dataframes.append(merged_df)

In [ ]:
# Concatenate per-image dataframes
final_df = pd.concat(dataframes, ignore_index=True)

# Save per-label results
if MIN_MAX_NUCLEI_VOLUME is not None:
    per_label_path = bp_results_dir / f"per_label_results_{input_folder_id}_MIN_{MIN_MAX_NUCLEI_VOLUME[0]}_MAX{MIN_MAX_NUCLEI_VOLUME[1]}_CT{CYTOPLASM_THICKNESS}.csv"
else:
    per_label_path = bp_results_dir / f"per_label_results_{input_folder_id}_MIN_None_MAX_None_CT{CYTOPLASM_THICKNESS}.csv"
final_df.to_csv(per_label_path, index=False)
per_label_path

In [ ]:
# Discard in memory Dataframe and read from disk (.csv copy)
final_df = pd.read_csv(per_label_path)
final_df.head()

In [ ]:
# Filter out rows where area_nuclei is less than 150 or greater than 2000
final_df_filtered = final_df[(final_df["area_nuclei"] >= 250) & (final_df["area_nuclei"] <= 1500)]

In [ ]:
# Group by organoid identifiers and compute per-organoid averages
grouped_df = final_df_filtered.groupby(["experiment_id", "mouse_id", "treatment_id", "replica_id"]).agg({
    "intensity_mean_nuclei": "mean",
    "area_nuclei": "mean",
    "intensity_mean_cyto": "mean",
    "area_cyto": "mean",
    "nuclei_cyto_ratio": "mean",
}).reset_index()

# Save average results
if MIN_MAX_NUCLEI_VOLUME is not None:
    average_path = bp_results_dir / f"average_nuclear_cyto_intensity_{input_folder_id}_MIN_{MIN_MAX_NUCLEI_VOLUME[0]}_MAX{MIN_MAX_NUCLEI_VOLUME[1]}_CT{CYTOPLASM_THICKNESS}.csv"
else:
    average_path = bp_results_dir / f"average_nuclear_cyto_intensity_{input_folder_id}_MIN_None_MAX_None_CT{CYTOPLASM_THICKNESS}.csv"
grouped_df.to_csv(average_path, index=False)
average_path

In [ ]:
# Discard in memory Dataframe and read from disk (.csv copy)
grouped_df = pd.read_csv(average_path)
grouped_df.head()

In [ ]:
# Create the box plot
fig = px.box(
    grouped_df,
    x="treatment_id",
    y="intensity_mean_nuclei",
    color="treatment_id",
    points="all",
    hover_data=["experiment_id", "mouse_id", "replica_id"],
    title="YAP Nuclear Intensity Average by Treatment ID",
)

fig.show()

In [ ]:
# Create the box plot
fig = px.box(
    grouped_df,
    x="treatment_id",
    y="intensity_mean_cyto",
    color="treatment_id",
    points="all",
    hover_data=["experiment_id", "mouse_id", "replica_id"],
    title="YAP Cytoplasmic Intensity Average by Treatment ID",
)

fig.show()

In [ ]:
# Create the box plot
fig = px.box(
    grouped_df,
    x="treatment_id",
    y="nuclei_cyto_ratio",
    color="treatment_id",
    points="all",
    hover_data=["experiment_id", "mouse_id", "replica_id"],
    title="YAP Nuclear/Cytoplasmic signal Ratio by Treatment ID",
)

fig.show()

In [ ]:
# Group by organoid identifiers and compute per-organoid averages
grouped_bio_rep_df = grouped_df.groupby(["mouse_id", "treatment_id"]).agg({
    "intensity_mean_nuclei": "mean",
    "area_nuclei": "mean",
    "intensity_mean_cyto": "mean",
    "area_cyto": "mean",
    "nuclei_cyto_ratio": "mean",
}).reset_index()

# Drop any row where 'treatment_id' contains the string "dmPGE2"
grouped_bio_rep_df = grouped_bio_rep_df[~grouped_bio_rep_df["treatment_id"].str.contains("dmPGE2", na=False)]
grouped_bio_rep_df

In [ ]:
import plotly.graph_objects as go

# Define the custom order for treatment_id
treatment_order = ["BCM", "MSN", "PGE2", "BCM_60min", "MSN_60min", "PGE2_60min"]

# Calculate means for each treatment in order
means = (
    grouped_bio_rep_df
    .set_index('treatment_id')
    .loc[treatment_order]  # preserve order
    .reset_index()
    [["treatment_id", "nuclei_cyto_ratio"]]
    .groupby("treatment_id")["nuclei_cyto_ratio"]
    .mean()
    .reindex(treatment_order)
    .values
)

# Scatter plot for individual data points, enforcing the order on x
fig = go.Figure()

for treatment in treatment_order:
    sub_df = grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == treatment]
    fig.add_trace(
        go.Scatter(
            x=[treatment] * len(sub_df),
            y=sub_df["nuclei_cyto_ratio"],
            mode="markers",
            name=treatment,
            text=sub_df["mouse_id"],
            marker=dict(size=10),
            showlegend=False  # disables individual duplicate legend entries
        )
    )

# Add mean as individual (not connected) diamond points for each treatment
for treatment, mean in zip(treatment_order, means):
    fig.add_trace(
        go.Scatter(
            x=[treatment],
            y=[mean],
            mode="markers",
            name="Mean" if treatment == treatment_order[0] else None,
            marker=dict(
                symbol="line-ew",
                size=28,
                color="black",
                line=dict(width=3, color="black"),
            ),

            showlegend=(treatment == treatment_order[0]),  # Only show legend entry once
        )
    )

fig.update_layout(
    xaxis=dict(title="treatment_id", categoryorder="array", categoryarray=treatment_order),
    yaxis_title="nuclei_cyto_ratio",
    title="YAP Nuclear/Cytoplasmic signal Ratio by Treatment ID (Mean ± Individual Points)",
)

fig.show()

In [ ]:
import scipy.stats as stats

# Define comparison groups
ttest_comparisons = [
    ("MSN", "BCM"),
    ("PGE2", "BCM"),
    ("MSN_60min", "BCM_60min"),
    ("PGE2_60min", "BCM_60min"),
]

anova_groups = [
    (["BCM", "MSN", "PGE2"], "BCM"),
    (["BCM_60min", "MSN_60min", "PGE2_60min"], "BCM_60min"),
]

print("==== T-TEST COMPARISONS BETWEEN GROUPS (nuclei_cyto_ratio) ====")
for case, ctrl in ttest_comparisons:
    case_vals = grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == case]["nuclei_cyto_ratio"]
    ctrl_vals = grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == ctrl]["nuclei_cyto_ratio"]
    tstat, pval = stats.ttest_ind(case_vals, ctrl_vals, equal_var=False, nan_policy='omit')
    print(f"{case} vs {ctrl}: t={tstat:.3f}, p={pval:.4g}")

print("\n==== 1-WAY ANOVA AND PAIRWISE COMPARISONS (nuclei_cyto_ratio) ====")
for groups, control in anova_groups:
    # Collect data for all groups
    group_vals = [grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == g]["nuclei_cyto_ratio"].dropna() for g in groups]
    anova_stat, anova_p = stats.f_oneway(*group_vals)
    print(f"ANOVA for {groups}: F={anova_stat:.3f}, p={anova_p:.4g}")
    # Pairwise t-test for each group vs control
    for grp in groups:
        if grp == control:
            continue
        grp_vals = grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == grp]["nuclei_cyto_ratio"]
        ctrl_vals = grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == control]["nuclei_cyto_ratio"]
        tstat, pval = stats.ttest_ind(grp_vals, ctrl_vals, equal_var=False, nan_policy='omit')
        print(f"  {grp} vs {control}: t={tstat:.3f}, p={pval:.4g}")